In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import joblib


In [6]:
from pathlib import Path

# Find the project root (the parent of the notebooks folder)
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

data_dir = project_dir / "data" / "cleaned"

accounts = pd.read_csv(data_dir / "ravenstack_accounts_clean.csv")
subscriptions = pd.read_csv(data_dir / "ravenstack_subscriptions_clean.csv")
feature_usage = pd.read_csv(data_dir / "ravenstack_feature_usage_clean.csv")
support_tickets = pd.read_csv(data_dir / "ravenstack_support_tickets_clean.csv")
churn_events = pd.read_csv(data_dir / "ravenstack_churn_events_clean.csv")
account_data = pd.read_csv(data_dir / "account_data.csv")

for df, columns in [
    (accounts, ["signup_date"]),
    (subscriptions, ["start_date", "end_date"]),
    (feature_usage, ["usage_date"]),
    (support_tickets, ["submitted_at", "closed_at"]),
    (churn_events, ["churn_date"])
]:
    for column in columns:
        df[column] = pd.to_datetime(df[column])

print("Datasets loaded successfully.")


print("dataset loaded")

Datasets loaded successfully.
dataset loaded


In [7]:
# Latest date available in the dataset
observation_end = max(
    feature_usage["usage_date"].max(),
    support_tickets["submitted_at"].max(),
    subscriptions["end_date"].dropna().max(),
    churn_events["churn_date"].max()
)

# First churn date only for customers whose final target is churn = 1
churned_accounts = accounts.loc[
    accounts["churn_flag"] == 1,
    "account_id"
]

account_churn_date = (
    churn_events[
        churn_events["account_id"].isin(churned_accounts)
    ]
    .groupby("account_id")["churn_date"]
    .min()
)

accounts["churn_date"] = accounts["account_id"].map(
    account_churn_date
)

# Default: use the end of the observation period
accounts["prediction_date"] = pd.Timestamp(observation_end)

# For actual churners: predict 30 days before churn
churned_mask = accounts["churn_flag"] == 1

accounts.loc[churned_mask, "prediction_date"] = (
    accounts.loc[churned_mask, "churn_date"]
    - pd.Timedelta(days=30)
)

print("Observation end:", observation_end)
print("\nPrediction dates created successfully.")

print("\nPrediction date examples:")
print(
    accounts[
        ["account_id", "churn_flag", "churn_date", "prediction_date"]
    ].head()
)

Observation end: 2024-12-31 00:00:00

Prediction dates created successfully.

Prediction date examples:
  account_id  churn_flag churn_date prediction_date
0   A-2e4581           0        NaT      2024-12-31
1   A-43a9e3           1        NaT             NaT
2   A-0a282f           0        NaT      2024-12-31
3   A-1f0ac7           0        NaT      2024-12-31
4   A-ce550d           1 2024-12-28      2024-11-28


In [8]:
# Basic customer information
model_data = accounts[[
    "account_id",
    "industry",
    "country",
    "plan_tier",
    "seats",
    "is_trial",
    "churn_flag",
    "signup_date",
    "prediction_date"
]].copy()

# Customer tenure at prediction date
model_data["tenure_months"] = (
    model_data["prediction_date"] -
    model_data["signup_date"]
).dt.days / 30.44


# --------------------------------------------------
# 1. Subscription information before prediction date
# --------------------------------------------------

subscription_pre = subscriptions.merge(
    model_data[["account_id", "prediction_date"]],
    on="account_id",
    how="inner"
)

subscription_pre = subscription_pre[
    subscription_pre["start_date"]
    <= subscription_pre["prediction_date"]
]

subscription_summary = (
    subscription_pre
    .groupby("account_id")
    .agg(
        avg_mrr=("mrr_amount", "mean"),
        upgrade_count=("upgrade_flag", "sum"),
        downgrade_count=("downgrade_flag", "sum"),
        auto_renew_count=("auto_renew_flag", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 2. Feature usage before prediction date
# --------------------------------------------------

usage_pre = feature_usage.merge(
    subscriptions[["subscription_id", "account_id"]],
    on="subscription_id",
    how="left"
)

usage_pre = usage_pre.merge(
    model_data[["account_id", "prediction_date"]],
    on="account_id",
    how="inner"
)

usage_pre = usage_pre[
    usage_pre["usage_date"]
    <= usage_pre["prediction_date"]
]

usage_summary = (
    usage_pre
    .groupby("account_id")
    .agg(
        total_usage=("usage_count", "sum"),
        total_usage_duration=("usage_duration_secs", "sum"),
        total_errors=("error_count", "sum"),
        beta_feature_usage=("is_beta_feature", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 3. Support information before prediction date
# --------------------------------------------------

support_pre = support_tickets.merge(
    model_data[["account_id", "prediction_date"]],
    on="account_id",
    how="inner"
)

support_pre = support_pre[
    support_pre["submitted_at"]
    <= support_pre["prediction_date"]
]

support_summary = (
    support_pre
    .groupby("account_id")
    .agg(
        total_tickets=("ticket_id", "count"),
        avg_resolution_hours=("resolution_time_hours", "mean"),
        avg_response_minutes=("first_response_time_minutes", "mean"),
        avg_satisfaction=("satisfaction_score", "mean"),
        escalated_tickets=("escalation_flag", "sum")
    )
    .reset_index()
)


# --------------------------------------------------
# 4. Combine all information
# --------------------------------------------------

model_data = (
    model_data
    .merge(
        subscription_summary,
        on="account_id",
        how="left"
    )
    .merge(
        usage_summary,
        on="account_id",
        how="left"
    )
    .merge(
        support_summary,
        on="account_id",
        how="left"
    )
)

print("Model dataset shape:", model_data.shape)
model_data.head()

Model dataset shape: (500, 23)


,account_id,industry,country,plan_tier,seats,is_trial,churn_flag,signup_date,prediction_date,tenure_months,...,auto_renew_count,total_usage,total_usage_duration,total_errors,beta_feature_usage,total_tickets,avg_resolution_hours,avg_response_minutes,avg_satisfaction,escalated_tickets
0,A-2e4581,EdTech,United States,Basic,9,0,0,2024-10-16,2024-12-31,2.496715,...,8.0,535.0,152339.0,38.0,4.0,2.0,23.000000,91.000000,3.000000,0.0
1,A-43a9e3,FinTech,India,Basic,18,0,1,2023-08-17,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A-0a282f,DevTools,United States,Basic,1,0,0,2024-08-27,2024-12-31,4.139290,...,15.0,821.0,251210.0,48.0,5.0,3.0,43.666667,63.666667,4.666667,0.0
3,A-1f0ac7,HealthTech,United Kingdom,Basic,24,1,0,2023-08-27,2024-12-31,16.162943,...,5.0,382.0,102528.0,21.0,5.0,2.0,29.000000,174.000000,NaN,0.0
4,A-ce550d,HealthTech,United States,Enterprise,35,0,1,2024-10-27,2024-11-28,1.051248,...,3.0,559.0,209127.0,28.0,4.0,7.0,42.285714,107.857143,3.800000,1.0


In [9]:
target = "churn_flag"

features = [
    "plan_tier",
    "industry",
    "country",
    "seats",
    "is_trial",
    "avg_mrr",
    "upgrade_count",
    "downgrade_count",
    "auto_renew_count",
    "total_usage",
    "total_usage_duration",
    "total_errors",
    "beta_feature_usage",
    "total_tickets",
    "avg_resolution_hours",
    "avg_response_minutes",
    "avg_satisfaction",
    "escalated_tickets",
    "tenure_months"
]

X = model_data[features]
y = model_data[target]

print("Number of features:", len(features))
print("Target distribution:")
print(y.value_counts())

Number of features: 19
Target distribution:
churn_flag
0    390
1    110
Name: count, dtype: int64


In [10]:
categorical_features = X.select_dtypes(
    include="object"
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['plan_tier', 'industry', 'country']

Numerical features:
['seats', 'is_trial', 'avg_mrr', 'upgrade_count', 'downgrade_count', 'auto_renew_count', 'total_usage', 'total_usage_duration', 'total_errors', 'beta_feature_usage', 'total_tickets', 'avg_resolution_hours', 'avg_response_minutes', 'avg_satisfaction', 'escalated_tickets', 'tenure_months']


In [11]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_transformer,
        numerical_features
    ),
    (
        "cat",
        categorical_transformer,
        categorical_features
    )
])

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


In [12]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples:", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nValidation target distribution:")
print(y_val.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training samples: 300
Validation samples: 100
Testing samples: 100

Training target distribution:
churn_flag
0    234
1     66
Name: count, dtype: int64

Validation target distribution:
churn_flag
0    78
1    22
Name: count, dtype: int64

Testing target distribution:
churn_flag
0    78
1    22
Name: count, dtype: int64


In [13]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (300, 31)
Processed testing shape: (100, 31)


In [14]:
logistic_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(
    X_train_processed,
    y_train
)

print("Balanced Logistic Regression trained.")

Balanced Logistic Regression trained.


c:\Users\Pratham\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [15]:
random_forest_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42
)

random_forest_model.fit(
    X_train_processed,
    y_train
)

print("Balanced Random Forest trained.")

Balanced Random Forest trained.


In [16]:
gradient_boosting_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=5,
    random_state=42
)

gradient_boosting_model.fit(
    X_train_processed,
    y_train
)

print("Gradient Boosting trained.")

Gradient Boosting trained.


In [17]:
def evaluate_model(model, X_eval_processed, y_eval, threshold=0.5):
    probabilities = model.predict_proba(X_eval_processed)[:, 1]
    predictions = (probabilities >= threshold).astype(int)

    return {
        "Accuracy": accuracy_score(y_eval, predictions),
        "Precision": precision_score(y_eval, predictions, zero_division=0),
        "Recall": recall_score(y_eval, predictions, zero_division=0),
        "F1": f1_score(y_eval, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, probabilities)
    }

In [18]:
model_results = pd.DataFrame({
    "Logistic Regression": evaluate_model(
        logistic_model, X_val_processed, y_val
    ),
    "Random Forest": evaluate_model(
        random_forest_model, X_val_processed, y_val
    ),
    "Gradient Boosting": evaluate_model(
        gradient_boosting_model, X_val_processed, y_val
    )
}).T

model_results.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC
Logistic Regression,0.71,0.370,0.455,0.408,0.686
Random Forest,0.85,0.818,0.409,0.545,0.810
Gradient Boosting,0.90,0.929,0.591,0.722,0.805


In [19]:
gb_predictions = gradient_boosting_model.predict(
    X_test_processed
)

print(
    classification_report(
        y_test,
        gb_predictions,
        target_names=["Retained", "Churned"],
        zero_division=0
    )
)

              precision    recall  f1-score   support

    Retained       0.91      0.96      0.94        78
     Churned       0.83      0.68      0.75        22

    accuracy                           0.90       100
   macro avg       0.87      0.82      0.84       100
weighted avg       0.90      0.90      0.90       100



In [20]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

gb_pipeline = Pipeline([
    ("preprocessor", ColumnTransformer([
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ])),
    ("model", GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=2,
        min_samples_leaf=5,
        random_state=42
    ))
])

cv_results = cross_validate(
    gb_pipeline,
    X,
    y,
    cv=cv,
    scoring=[
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc"
    ]
)

print("Cross-validation results:\n")

for metric in [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]:
    scores = cv_results[f"test_{metric}"]

    print(
        f"{metric.upper():10}: "
        f"{scores.mean():.3f} ± {scores.std():.3f}"
    )

Cross-validation results:

ACCURACY  : 0.926 ± 0.015
PRECISION : 0.924 ± 0.051
RECALL    : 0.727 ± 0.076
F1        : 0.810 ± 0.043
ROC_AUC   : 0.916 ± 0.033


In [21]:
print("Validation-set model comparison:")
print(model_results.round(3))

print("\nGradient Boosting cross-validation:")
print(pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
    "CV Mean": [
        cv_results["test_accuracy"].mean(),
        cv_results["test_precision"].mean(),
        cv_results["test_recall"].mean(),
        cv_results["test_f1"].mean(),
        cv_results["test_roc_auc"].mean()
    ]
}).round(3))

Validation-set model comparison:
                     Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression      0.71      0.370   0.455  0.408    0.686
Random Forest            0.85      0.818   0.409  0.545    0.810
Gradient Boosting        0.90      0.929   0.591  0.722    0.805

Gradient Boosting cross-validation:
      Metric  CV Mean
0   Accuracy    0.926
1  Precision    0.924
2     Recall    0.727
3         F1    0.810
4    ROC-AUC    0.916


In [22]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.05,
        max_depth=2,
        min_samples_leaf=5,
        random_state=42
    ))
])

final_model.fit(X_train, y_train)

print("Final Gradient Boosting model trained on training data.")

Final Gradient Boosting model trained on training data.


In [23]:
feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
model = final_model.named_steps["model"]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance.head(15))

,feature,importance
2,num__avg_mrr,0.527884
5,num__auto_renew_count,0.151750
15,num__tenure_months,0.146204
7,num__total_usage_duration,0.043798
8,num__total_errors,0.026664
3,num__upgrade_count,0.022841
0,num__seats,0.020328
10,num__total_tickets,0.018757
12,num__avg_response_minutes,0.012127
6,num__total_usage,0.007676


In [24]:
# Feature importance from the fitted Gradient Boosting pipeline
feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
model = final_model.named_steps["model"]

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("Top 15 features influencing churn prediction:")
display(feature_importance.head(15))

Top 15 features influencing churn prediction:


,feature,importance
2,num__avg_mrr,0.527884
5,num__auto_renew_count,0.151750
15,num__tenure_months,0.146204
7,num__total_usage_duration,0.043798
8,num__total_errors,0.026664
3,num__upgrade_count,0.022841
0,num__seats,0.020328
10,num__total_tickets,0.018757
12,num__avg_response_minutes,0.012127
6,num__total_usage,0.007676


In [25]:
feature_importance["feature"] = (
    feature_importance["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
    .str.replace("_", " ", regex=False)
)

feature_importance["feature"] = (
    feature_importance["feature"]
    .str.title()
)

feature_importance.head(15)

,feature,importance
2,Avg Mrr,0.527884
5,Auto Renew Count,0.151750
15,Tenure Months,0.146204
7,Total Usage Duration,0.043798
8,Total Errors,0.026664
3,Upgrade Count,0.022841
0,Seats,0.020328
10,Total Tickets,0.018757
12,Avg Response Minutes,0.012127
6,Total Usage,0.007676


In [26]:
top_features = feature_importance.head(10).copy()

print("Top 10 features influencing churn:\n")
print(top_features)

Top 10 features influencing churn:

                 feature  importance
2                Avg Mrr    0.527884
5       Auto Renew Count    0.151750
15         Tenure Months    0.146204
7   Total Usage Duration    0.043798
8           Total Errors    0.026664
3          Upgrade Count    0.022841
0                  Seats    0.020328
10         Total Tickets    0.018757
12  Avg Response Minutes    0.012127
6            Total Usage    0.007676


In [27]:
top_features.to_csv(
    "feature_importance.csv",
    index=False
)

print("Feature importance saved successfully.")

Feature importance saved successfully.


In [28]:
gb_val_probabilities = final_model.predict_proba(X_val)[:, 1]

threshold_values = np.arange(0.20, 0.71, 0.05)

threshold_results = []

for threshold in threshold_values:
    val_predictions = (
        gb_val_probabilities >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": round(threshold, 2),
        "Accuracy": accuracy_score(y_val, val_predictions),
        "Precision": precision_score(
            y_val, val_predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_val, val_predictions, zero_division=0
        ),
        "F1": f1_score(
            y_val, val_predictions, zero_division=0
        )
    })

threshold_results = pd.DataFrame(threshold_results)

display(threshold_results.round(3))

best_threshold = threshold_results.loc[
    threshold_results["F1"].idxmax(),
    "Threshold"
]

print("Best validation threshold:", best_threshold)

,Threshold,Accuracy,Precision,Recall,F1
0,0.20,0.85,0.640,0.727,0.681
1,0.25,0.86,0.682,0.682,0.682
2,0.30,0.89,0.789,0.682,0.732
3,0.35,0.90,0.875,0.636,0.737
4,0.40,0.91,0.933,0.636,0.757
5,0.45,0.91,0.933,0.636,0.757
6,0.50,0.90,0.929,0.591,0.722
7,0.55,0.88,0.917,0.500,0.647
8,0.60,0.88,0.917,0.500,0.647
9,0.65,0.88,0.917,0.500,0.647


Best validation threshold: 0.4


In [29]:
gb_probabilities = final_model.predict_proba(X_test)[:, 1]

final_test_predictions = (
    gb_probabilities >= best_threshold
).astype(int)

final_metrics = {
    "Accuracy": accuracy_score(y_test, final_test_predictions),
    "Precision": precision_score(
        y_test, final_test_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_test, final_test_predictions, zero_division=0
    ),
    "F1": f1_score(
        y_test, final_test_predictions, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(y_test, gb_probabilities)
}

print("Final test performance:\n")

for metric, value in final_metrics.items():
    print(f"{metric}: {value:.3f}")

Final test performance:

Accuracy: 0.900
Precision: 0.800
Recall: 0.727
F1: 0.762
ROC-AUC: 0.876


In [30]:
joblib.dump(final_model, "churn_model.pkl")
joblib.dump(best_threshold, "churn_threshold.pkl")

print("Complete model pipeline saved successfully.")
print("Selected threshold:", best_threshold)

Complete model pipeline saved successfully.
Selected threshold: 0.4


In [31]:
print("Selected validation threshold:", best_threshold)
print("Untouched test samples:", len(y_test))
print("Actual churned:", int(y_test.sum()))

print("\nFinal test confusion matrix:")
print(confusion_matrix(y_test, final_test_predictions))

print("\nTest ROC-AUC:")
print(roc_auc_score(y_test, gb_probabilities))

Selected validation threshold: 0.4
Untouched test samples: 100
Actual churned: 22

Final test confusion matrix:
[[74  4]
 [ 6 16]]

Test ROC-AUC:
0.8764568764568764


In [32]:
saved_model = joblib.load("churn_model.pkl")

print("Model type:", type(saved_model))
print("Pipeline steps:", saved_model.named_steps.keys())
print("Saved threshold:", joblib.load("churn_threshold.pkl"))

Model type: <class 'sklearn.pipeline.Pipeline'>
Pipeline steps: dict_keys(['preprocessor', 'model'])
Saved threshold: 0.4


In [33]:
import sys
import sklearn
import joblib

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

Python version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]
Python executable: c:\Users\Pratham\AppData\Local\Python\pythoncore-3.14-64\python.exe
Scikit-learn: 1.8.0
Joblib: 1.5.3


In [34]:
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

model_path = project_dir / "models" / "churn_model.pkl"
model = joblib.load(model_path)

print("Model loaded:", type(model))
print("Pipeline steps:", model.named_steps.keys())

Model loaded: <class 'sklearn.pipeline.Pipeline'>
Pipeline steps: dict_keys(['preprocessor', 'model'])


In [ ]:

model_path = Path("../models/churn_model.pkl")

print("Scikit-learn:", sklearn.__version__)
print("Model exists:", model_path.exists())

model = joblib.load(model_path)
print("Model type:", type(model))

Scikit-learn: 1.8.0
Model exists: True
Model type: <class 'sklearn.pipeline.Pipeline'>


In [36]:
variables = [
    "X_test", "y_test",
    "X_train", "y_train",
    "final_model", "best_model",
    "final_test_predictions", "y_pred",
    "y_test_proba", "y_pred_proba",
    "threshold", "best_threshold"
]

for name in variables:
    if name in globals():
        obj = globals()[name]
        print(
            f"{name}: {type(obj).__name__}",
            f"| shape={obj.shape}" if hasattr(obj, "shape") else ""
        )

X_test: DataFrame | shape=(100, 19)
y_test: Series | shape=(100,)
X_train: DataFrame | shape=(300, 19)
y_train: Series | shape=(300,)
final_model: Pipeline 
final_test_predictions: ndarray | shape=(100,)
threshold: float64 | shape=()
best_threshold: float64 | shape=()


In [ ]:
# 1. Generate churn probabilities
test_probabilities = final_model.predict_proba(X_test)[:, 1]

# 2. Apply the selected threshold
threshold_value = float(best_threshold)
test_predictions = (test_probabilities >= threshold_value).astype(int)

# 3. Calculate evaluation metrics
test_metrics = {
    "Accuracy": accuracy_score(y_test, test_predictions),
    "Precision": precision_score(y_test, test_predictions, zero_division=0),
    "Recall": recall_score(y_test, test_predictions, zero_division=0),
    "F1": f1_score(y_test, test_predictions, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, test_probabilities)
}

print(f"Threshold: {threshold_value:.3f}")
print("\nTest-set performance:")
display(pd.DataFrame([test_metrics]).round(3))

print("\nClassification report:")
print(classification_report(
    y_test,
    test_predictions,
    labels=[0, 1],
    target_names=["Retained", "Churned"],
    zero_division=0
))

# 4. Generate the confusion matrix
cm = confusion_matrix(y_test, test_predictions, labels=[0, 1])

cm_df = pd.DataFrame(
    cm,
    index=["Actual Retained", "Actual Churned"],
    columns=["Predicted Retained", "Predicted Churned"]
)

print("\nConfusion matrix:")
display(cm_df)

Threshold: 0.400

Test-set performance:


,Accuracy,Precision,Recall,F1,ROC-AUC
0,0.9,0.8,0.727,0.762,0.876



Classification report:
              precision    recall  f1-score   support

    Retained       0.93      0.95      0.94        78
     Churned       0.80      0.73      0.76        22

    accuracy                           0.90       100
   macro avg       0.86      0.84      0.85       100
weighted avg       0.90      0.90      0.90       100


Confusion matrix:


,Predicted Retained,Predicted Churned
Actual Retained,74,4
Actual Churned,6,16


In [39]:
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

output_dir = project_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Save confusion matrix
cm_df.to_csv(output_dir / "confusion_matrix.csv")

# Save final test metrics
pd.DataFrame([test_metrics]).to_csv(
    output_dir / "test_metrics.csv",
    index=False
)

print("Saved successfully:")
print(output_dir / "confusion_matrix.csv")
print(output_dir / "test_metrics.csv")

Saved successfully:
c:\Users\Pratham\OneDrive\Desktop\Saas_project_details\outputs\confusion_matrix.csv
c:\Users\Pratham\OneDrive\Desktop\Saas_project_details\outputs\test_metrics.csv
